In [0]:
%run ./utils_api_client

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 99 Utils — Câmara API Pagination
# MAGIC
# MAGIC **Notebook:** `utils_pagination`
# MAGIC
# MAGIC Provides reusable pagination functions for Câmara dos Deputados Open Data API endpoints.
# MAGIC
# MAGIC This notebook preserves a simple and resilient pagination strategy because the API may present intermittent instability, empty pages, timeout errors or inconsistent endpoint behavior.
# MAGIC
# MAGIC ## Responsibilities
# MAGIC - Control page-based API extraction
# MAGIC - Apply `pagina` and `itens` parameters
# MAGIC - Reuse centralized API request function
# MAGIC - Support retry behavior for unstable endpoints
# MAGIC - Stop pagination safely when no records are returned
# MAGIC - Support execution limits for tests and controlled ingestion
# MAGIC
# MAGIC ## Technical Notes
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and technical documentation are written in English.
# MAGIC - The implementation intentionally keeps direct loop logic for operational stability.

# COMMAND ----------

from typing import Optional, Dict, Any, List
import time

# COMMAND ----------

# MAGIC %run ./utils_config

# COMMAND ----------

# MAGIC %run ./utils_api_client

# COMMAND ----------

def collect_pages(
    endpoint_path: str,
    base_params: Optional[Dict[str, Any]] = None,
    record_limit: Optional[int] = None,
    page_size: Optional[int] = None,
    request_timeout: Optional[int] = None,
    max_retries: Optional[int] = None,
    sleep_seconds: float = 0.5,
) -> List[Dict[str, Any]]:
    """
    Collects records from a paginated Câmara dos Deputados API endpoint.
    """

    if page_size is None:
        page_size = API_DEFAULT_PAGE_SIZE

    if request_timeout is None:
        request_timeout = API_REQUEST_TIMEOUT_SECONDS

    if max_retries is None:
        max_retries = API_MAX_RETRY_ATTEMPTS

    collected_records = []
    page_number = 1

    while True:

        page_params = dict(base_params or {})
        page_params[API_PAGE_PARAMETER_NAME] = page_number
        page_params[API_PAGE_SIZE_PARAMETER_NAME] = page_size

        last_error = None
        page_records = []

        for retry_number in range(1, max_retries + 1):

            try:
                api_response = fetch_camara_api_data(
                    endpoint_path=endpoint_path,
                    query_params=page_params,
                    request_timeout_seconds=request_timeout,
                    max_retry_attempts=1,
                )

                page_records = api_response.get(API_RESPONSE_DATA_FIELD, [])
                break

            except Exception as error:
                last_error = error

                print(
                    f"[WARNING] Pagination request failed "
                    f"| endpoint={endpoint_path} "
                    f"| page={page_number} "
                    f"| attempt={retry_number}/{max_retries} "
                    f"| error={str(error)}"
                )

                if retry_number == max_retries:
                    raise last_error

                time.sleep(sleep_seconds * retry_number)

        if not page_records:
            print(
                f"[INFO] Pagination finished with empty response "
                f"| endpoint={endpoint_path} "
                f"| page={page_number}"
            )
            break

        collected_records.extend(page_records)

        print(
            f"[INFO] Page collected "
            f"| endpoint={endpoint_path} "
            f"| page={page_number} "
            f"| page_records={len(page_records)} "
            f"| total_records={len(collected_records)}"
        )

        if len(page_records) < page_size:
            print(
                f"[INFO] Pagination finished with partial last page "
                f"| endpoint={endpoint_path} "
                f"| page={page_number}"
            )
            break

        if record_limit is not None and len(collected_records) >= record_limit:
            print(
                f"[INFO] Pagination stopped by record limit "
                f"| endpoint={endpoint_path} "
                f"| limit={record_limit}"
            )
            break

        page_number += 1

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    if record_limit is not None:
        return collected_records[:record_limit]

    return collected_records

# COMMAND ----------

def collect_first_page(
    endpoint_path: str,
    base_params: Optional[Dict[str, Any]] = None,
    page_size: Optional[int] = None,
    request_timeout: Optional[int] = None,
) -> List[Dict[str, Any]]:
    """
    Collects only the first page from a Câmara API endpoint.
    """

    if page_size is None:
        page_size = API_DEFAULT_PAGE_SIZE

    if request_timeout is None:
        request_timeout = API_REQUEST_TIMEOUT_SECONDS

    first_page_params = dict(base_params or {})
    first_page_params[API_PAGE_PARAMETER_NAME] = 1
    first_page_params[API_PAGE_SIZE_PARAMETER_NAME] = page_size

    api_response = fetch_camara_api_data(
        endpoint_path=endpoint_path,
        query_params=first_page_params,
        request_timeout_seconds=request_timeout,
    )

    return api_response.get(API_RESPONSE_DATA_FIELD, [])

# COMMAND ----------

print("utils_pagination loaded successfully.")

utils_pagination loaded successfully.


Project configuration loaded successfully.
Project: brazil_legislative_analytics
Catalog: brazil_legislative_analytics
Environment: dev
Default legislatures: [56, 57]
Analysis years: [2022, 2023, 2024, 2025, 2026]


utils_api_client loaded successfully.
